# NB39 — Per-KO Niche Breadth and Geochemical Diversity PGLS

Two analyses extending NB37–38 niche breadth work:

**Option A**: Per-KO L0a PGLS — does any individual Tier 1/2 KO drive the L0a habitat niche breadth signal?

**Option B**: Multi-layer niche breadth PGLS — do alternative niche dimensions (geochemical, edaphic, climatic, anthropogenic) predict metal gene density, and in what direction?

**Key finding**:
- Option A: 0/118 Tier 1/2 KOs are FDR-significant — the L0a signal (β=−0.517***) is distributed, not driven by any single gene
- Option B: SoilGrids CEC breadth is **positively** associated with metal gene density (β=+0.590***) — sign reversal from L0a. Genera that span broad CEC environments have *more* metal genes.

In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = '1'

import sys
sys.path.insert(0, '/home/hmacgregor/BERIL-research-observatory/tools')
from figure_style import apply_style, save, PALETTE, METAL_COLORS, FIGW, ROW_H
apply_style()

sys.path.insert(0, '/home/hmacgregor/BERIL-research-observatory/projects/comprehensive_metal_ecology/scripts')
from scripts.pgls_utils import run_pgls

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

DATA = Path('../data')
FIGS = Path('../figures')
TREE_BAC = DATA / 'gtdb_bac_genus_pruned.tree'

p1 = pd.read_csv(DATA / '01_pgls_input_bacteria.csv')
p1['genus_lower'] = p1['genus_lower'].str.lower()
base = p1[['genus_lower', 'ko_per_mb_primary', 'mean_levins_B_std']].dropna()

def z(s): return (s - s.mean()) / s.std()

## Option A — Per-KO L0a Niche Breadth PGLS

For each of the 118 Tier 1/2 KOs: compute per-genus KO density (proportion of genomes with KO / mean genome Mb), z-score, and run PGLS vs L0a niche breadth (mean_levins_B_std). Apply FDR-BH correction.

**Pre-computed results** in `data/39_per_ko_levinsB_pgls.csv`.

In [ ]:
ko_pgls = pd.read_csv(DATA / '39_per_ko_levinsB_pgls.csv')

print(f'KOs tested: {len(ko_pgls)}')
print(f'FDR-significant (q<0.05): {(ko_pgls["q"] < 0.05).sum()}')
print(f'FDR-significant (q<0.20): {(ko_pgls["q"] < 0.20).sum()}')
print()
print('Top 10 by nominal p:')
top10 = ko_pgls.nsmallest(10, 'p')[['ko', 'subcategory', 'tier', 'beta', 'SE', 'p', 'q', 'n', 'sig']]
print(top10.to_string(index=False))

**Result**: 0/118 KOs pass FDR correction. The L0a niche breadth signal (β=−0.517*** at the aggregate level) is distributed uniformly across all metal gene categories — no single KO drives it.

In [ ]:
# Subcategory-level summary of nominal directionality
sub_summary = ko_pgls.groupby('subcategory').agg(
    n_kos=('ko', 'count'),
    n_negative=('beta', lambda x: (x < 0).sum()),
    n_positive=('beta', lambda x: (x > 0).sum()),
    median_beta=('beta', 'median'),
    n_nominal_p05=('p', lambda x: (x < 0.05).sum())
).reset_index()
print('Subcategory directionality:')
print(sub_summary.to_string(index=False))

In [ ]:
# Figure: per-KO beta distribution by subcategory
cat_order = ko_pgls.groupby('subcategory')['beta'].median().sort_values().index.tolist()
colors = {cat: PALETTE[i % len(PALETTE)] for i, cat in enumerate(cat_order)}

fig, ax = plt.subplots(figsize=(FIGW['2col'], ROW_H))

for i, cat in enumerate(cat_order):
    sub = ko_pgls[ko_pgls['subcategory'] == cat]
    y = np.random.normal(i, 0.1, len(sub))
    ax.scatter(sub['beta'], y, color=colors[cat], alpha=0.7, s=25, zorder=3)

ax.axvline(0, color='gray', lw=0.8, ls='--')
ax.axvline(-0.517, color='darkred', lw=1.0, ls=':', alpha=0.7, label='L0a aggregate β=−0.517')
ax.set_yticks(range(len(cat_order)))
ax.set_yticklabels(cat_order, fontsize=8)
ax.set_xlabel('PGLS β (KO density ~ L0a niche breadth)')
ax.set_ylabel('')
ax.set_title('Per-KO PGLS: L0a niche breadth (0/118 FDR-sig)', fontsize=10)
ax.legend(fontsize=8, loc='upper right')
ax.annotate(f'n = 118 KOs', xy=(0.98, 0.02), xycoords='axes fraction',
            ha='right', va='bottom', fontsize=8, color='#808080')

plt.tight_layout()
save(fig, FIGS / 'fig_nb39_per_ko_beta_distribution')
print('Saved fig_nb39_per_ko_beta_distribution.pdf')

## Option B — Multi-Layer Niche Breadth PGLS

Tests whether alternative niche dimensions predict metal gene density.

**Layers available from existing parquets (Spark runs from NB38):**
- `KG_climate_all`: Köppen-Geiger climate classes (Arid/Continental/Polar; 3 classes)
- `PopDens_all`: population density quintile breadth (5 classes; all biomes)
- `PopDens_freshwater`: population density breadth (freshwater-only genera)
- `SoilGrids_CEC`: cation exchange capacity class breadth (5 classes: very_low → very_high)
- `SoilGrids_Clay`: clay textural class breadth (4 classes: sand → clay)
- `NGSA_geochm_AUS`: mean per-genus metal SD across Australian NGSA sampling sites (geochemical niche breadth)
- `L0a_ref_habitats`: habitat type Levins B (reference from NB37)

**Note**: ESA CCI land cover parquets are empty (Spark write failed for MicrobeAtlas × ESA CCI join). Landcover-biome-stratified results are unavailable without a Spark re-run.

In [ ]:
bs = pd.read_csv(DATA / '39_biome_stratified_pgls.csv')
bs['ci_lo'] = bs['beta'] - 1.96 * bs['SE']
bs['ci_hi'] = bs['beta'] + 1.96 * bs['SE']
bs = bs.sort_values('beta')
print('Biome-stratified PGLS results:')
print(bs[['layer','beta','SE','p','n','sig']].to_string(index=False))

**Key results:**
1. **SoilGrids_CEC** (β=+0.590***): Strong *positive* association — genera spanning more CEC classes have *more* metal genes. Sign reversal from L0a.
2. **L0a** (β=−0.517***): Habitat diversity predicts fewer metal genes (NB37 reference).
3. **KG_climate** (β=+0.190*): Marginally positive — genera spanning Arid/Continental/Polar climate zones have slightly more metal genes.
4. **NGSA geochemical, PopDens, Clay**: All null.

**Interpretation of the CEC sign reversal**: CEC (meq/100g) governs metal bioavailability — high-CEC soils bind more metals as exchangeable cations. Genera found across high and low CEC soils must cope with dramatically different metal exposure conditions, requiring more metal homeostasis machinery. By contrast, L0a habitat diversity (soil/marine/freshwater/urban) measures ecological breadth, not geochemical challenge specifically. Habitat generalists achieve breadth via metabolic flexibility, not metal-gene proliferation.

In [ ]:
# Figure: multi-layer forest plot
layer_labels = {
    'L0a_ref_habitats': 'L0a habitat diversity\n(NB37 reference)',
    'SoilGrids_CEC': 'SoilGrids CEC classes\n(5 bins: very low→very high)',
    'KG_climate_all': 'Köppen-Geiger climate\n(Arid/Continental/Polar)',
    'SoilGrids_Clay': 'SoilGrids clay texture\n(4 classes)',
    'PopDens_freshwater': 'Population density\n(freshwater genera)',
    'PopDens_all': 'Population density\n(all biomes)',
    'NGSA_geochm_AUS': 'NGSA geochemical niche\n(Australia, ICP-MS SD)',
}
data_type = {
    'L0a_ref_habitats': 'Habitat/Climate',
    'SoilGrids_CEC': 'Edaphic',
    'KG_climate_all': 'Habitat/Climate',
    'SoilGrids_Clay': 'Edaphic',
    'PopDens_freshwater': 'Anthropogenic',
    'PopDens_all': 'Anthropogenic',
    'NGSA_geochm_AUS': 'Geochemical',
}
type_colors = {
    'Habitat/Climate': PALETTE[0],
    'Edaphic': PALETTE[2],
    'Anthropogenic': PALETTE[4],
    'Geochemical': PALETTE[1],
}

fig, ax = plt.subplots(figsize=(FIGW['2col'], ROW_H * 1.4))

bs_plot = bs.copy()
n_rows = len(bs_plot)
ys = list(range(n_rows))

for i, (_, row) in enumerate(bs_plot.iterrows()):
    layer = row['layer']
    color = type_colors.get(data_type.get(layer, 'Other'), 'gray')
    ax.errorbar(row['beta'], i, xerr=[[row['beta'] - row['ci_lo']], [row['ci_hi'] - row['beta']]],
                fmt='o', color=color, ecolor=color, elinewidth=1.2, capsize=3, ms=6, zorder=3)
    sig = row['sig']
    if sig != 'ns':
        ax.text(row['ci_hi'] + 0.02, i, sig, va='center', ha='left', fontsize=8, color=color)

ax.axvline(0, color='gray', lw=0.8, ls='--')
ax.set_yticks(ys)
ax.set_yticklabels([layer_labels.get(row['layer'], row['layer']) for _, row in bs_plot.iterrows()], fontsize=7.5)
ax.set_xlabel('PGLS β (ko_per_mb_primary ~ niche breadth index)')
ax.set_ylabel('')
ax.set_title('Multi-layer niche breadth PGLS: metal gene density', fontsize=10)

# Legend for data type colors
from matplotlib.patches import Patch
legend_handles = [Patch(color=c, label=t) for t, c in type_colors.items()]
ax.legend(handles=legend_handles, fontsize=7, loc='lower right', title='Niche dimension', title_fontsize=7)

ax.annotate(f'n ≈ 1,570–1,574 genera per test', xy=(0.98, 0.98), xycoords='axes fraction',
            ha='right', va='top', fontsize=8, color='#808080')

plt.tight_layout()
save(fig, FIGS / 'fig_nb39_multi_layer_niche_forest')
print('Saved fig_nb39_multi_layer_niche_forest.pdf')

## Summary and Interpretation

### Option A: No single KO drives the L0a signal
118 Tier 1/2 KOs tested individually. 0 pass FDR correction. The top nominal hit (K17225, Metal-dependent Metabolism, β=−0.232, q=0.127) barely approaches FDR threshold. The aggregate signal (β=−0.517***) emerges from the collective of metal genes, not from any dominant gene family.

### Option B: Two orthogonal niche dimensions, opposite signs

| Niche dimension | β | Sig | Direction |
|---|---|---|---|
| L0a habitat diversity | −0.517 | *** | Generalists have fewer metal genes |
| SoilGrids CEC class breadth | +0.590 | *** | Geochemically diverse → more metal genes |
| KG climate zone breadth | +0.190 | * | Multi-climate → slightly more metal genes |
| Clay texture class breadth | +0.076 | ns | — |
| Population density breadth | −0.033 | ns | — |
| NGSA geochemical (Australia) | +0.021 | ns | — |

**Biological interpretation**: L0a and CEC measure fundamentally different things:
- L0a quantifies **ecological breadth** across biome types (terrestrial/aquatic/built).
  Wide habitat range is a hallmark of *r*-strategists and copiotrophic generalists, which invest in genomic economy rather than metabolic specialization.
- CEC quantifies **soil chemical heterogeneity** — high-CEC soils (clay-rich, organic-rich) bind many metals as exchangeable cations; low-CEC soils (sandy, low-organic) do not.
  Genera found across the CEC spectrum experience highly variable metal bioavailability and must maintain a larger repertoire of metal homeostasis genes to survive across environments.

**These are not contradictory** — they are orthogonal axes:
- A habitat generalist that also spans CEC gradients: many habitats, but soil-specialist within those habitats → mixed signal
- A soil-specialist with narrow CEC range: few habitat types, low CEC breadth → low metal genes (consistent with L0a)
- A soil specialist spanning all CEC classes: few habitat types, high CEC breadth → more metal genes (consistent with CEC finding)

### Data gaps (flagged)
- **ESA CCI land cover parquets empty**: Spark join failed; re-run required for biome-stratified landcover L0a
- **USGS geochemical niche (USA)**: requires Spark join of MicrobeAtlas OTU lat/lon → USGS 0.5° grid; no local genus×site matrix
- **GEMAS (Europe)**: requires Spark access to `arkinlab.envdbs.gemas`
- **KG only 3 climate classes**: Tropical class absent (MicrobeAtlas dominated by temperate/cold environments)

## Section C — Redox-Controlled Sensitivity Analysis

The 10 significant hits from the extended 36-test univariate screen (GeoROC breadth, CSU bioavailability breadth, SILVA Levins B, GeoROC metal index level, mine distance, soil pH) are retested with `median_soil_moisture` as a redox proxy covariate. Soil moisture is the best globally-available redox proxy: n ≈ 9,959 genera, covering all P1 genera.

**Covariate model:** `ko_per_mb_primary ~ focal_predictor_z + soil_moisture_z`

Pre-computed results in `data/39_redox_controlled_pgls.csv`.

In [ ]:
rdx = pd.read_csv(DATA / '39_redox_controlled_pgls.csv')
print('Redox-controlled sensitivity results:')
print(rdx[['predictor','sig_univariate','sig_ctrl_moisture','verdict','note']].to_string(index=False))

n_survive = (rdx['sig_ctrl_moisture'] != 'ns').sum()
n_attenuated = (rdx['sig_ctrl_moisture'] == 'ns').sum()
print(f'\n{n_survive}/10 survive moisture control; {n_attenuated}/10 attenuated (sample-size reduction in merge)')

In [ ]:
# Figure: side-by-side forest plot (univariate vs moisture-controlled)
survive = rdx[rdx['sig_ctrl_moisture'] != 'ns'].copy()
attenuated = rdx[rdx['sig_ctrl_moisture'] == 'ns'].copy()

fig, ax = plt.subplots(figsize=(FIGW['2col'], ROW_H * 1.2))

# Sort by controlled beta
rdx_plot = rdx.copy()
rdx_plot['b_u'] = rdx_plot['beta_univariate'].astype(float)
rdx_plot['b_c'] = rdx_plot['beta_ctrl_moisture'].astype(float)
rdx_plot = rdx_plot.sort_values('b_u')

ys = list(range(len(rdx_plot)))
offset = 0.15  # vertical offset for paired dots

for i, (_, row) in enumerate(rdx_plot.iterrows()):
    is_sig_ctrl = row['sig_ctrl_moisture'] != 'ns'
    c_u = PALETTE[0]
    c_c = PALETTE[2] if is_sig_ctrl else 'gray'

    ax.scatter(row['b_u'], i + offset, color=c_u, s=35, zorder=4, label='Univariate' if i == 0 else '')
    ax.scatter(row['b_c'], i - offset, color=c_c, s=35, zorder=4,
               label='+ Soil moisture covariate' if i == 0 else '', marker='D')
    # connector
    ax.plot([row['b_u'], row['b_c']], [i + offset, i - offset],
            color='gray', lw=0.6, zorder=2, alpha=0.6)

ax.axvline(0, color='gray', lw=0.8, ls='--')
short_labels = [r['predictor'].replace(' — ', '\n').replace(' (level)', '\n(level)').replace(' (16S)', '\n(16S)')
                for _, r in rdx_plot.iterrows()]
ax.set_yticks(ys)
ax.set_yticklabels(short_labels, fontsize=7)
ax.set_xlabel('PGLS β (ko_per_mb_primary ~ predictor)')
ax.set_ylabel('')
ax.set_title('Redox sensitivity: univariate vs +soil moisture covariate', fontsize=10)
ax.legend(fontsize=7, loc='lower right')
ax.annotate('Gray/attenuated = non-significant after moisture control', xy=(0.98, 0.02),
            xycoords='axes fraction', ha='right', va='bottom', fontsize=7, color='#808080')

plt.tight_layout()
save(fig, FIGS / 'fig_nb39_redox_sensitivity')
print('Saved fig_nb39_redox_sensitivity.pdf')

### Redox sensitivity — key findings

**8/10 predictors survive redox control (soil moisture covariate):**

| Predictor | Univariate β | Controlled β | Verdict |
|---|---|---|---|
| GeoROC Cd breadth | +0.431*** | +0.431*** | SURVIVES — unchanged |
| CSU Hg bioavail breadth | +0.327*** | +0.335*** | SURVIVES — slightly stronger |
| GeoROC metal index (level) | −0.341*** | −0.341*** | SURVIVES |
| Mine distance (CMMI) | +0.303** | +0.299** | SURVIVES |
| GeoROC Zn breadth | +0.238** | +0.238** | SURVIVES |
| SILVA Levins B | −0.251** | −0.253** | SURVIVES — independent validation |
| **Soil pH (level)** | −0.224* | **−0.442****** | **SURVIVES — STRENGTHENED** |
| CSU Cr bioavail breadth | +0.205* | +0.204* | SURVIVES |
| GeoROC Co breadth | +0.113 ns | +0.117 ns | attenuated (n reduces in join) |
| GeoROC Pb breadth | +0.133 ns | +0.131 ns | attenuated (n reduces in join) |

**Key interpretation:**

- **Cd and Hg are redox-independent**: The geochemical breadth signals for these metals persist unchanged after moisture control. Cd methylation is not strongly redox-gated; Hg bioavailability (CSU fraction) reflects total bioavailable Hg rather than anaerobic Hg methylation. Metal gene carriage for these metals reflects geochemical exposure breadth, not redox habitat breadth.

- **Soil pH strengthens dramatically** (+0.224* → −0.442***): soil moisture and pH are positively correlated in nature (wet soils tend to be more acidic due to CO₂/organic acid production). By controlling for moisture, the residual pH effect is actually twice as large — pH itself (independent of moisture) is a dominant driver of metal bioavailability and metal gene carriage.

- **Co and Pb attenuate**: These were borderline significant in the full dataset. The inner join with moisture reduces genus N; the attenuation likely reflects sample size loss rather than true redox confounding. The betas barely change (Δ < 0.01).

- **CMMI (mine distance) survives**: β=+0.303** → +0.299** — the anthropogenic contamination proximity signal is not explained by differences in soil moisture near mines. More likely reflects geographic/vegetation effects.